In [2]:
gpt_config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}
from gpt_model import GPTModel

model = GPTModel(gpt_config)

In [7]:
import torch
import tiktoken

torch.manual_seed(42)

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text):
    tokens = tokenizer.encode(
        text = text,
        allowed_special = {'<|endoftext|>'}
        )
    return torch.tensor(tokens).unsqueeze(0)

def token_ids_to_text(token_ids):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [25]:
import torch
def generate_text(model, context, max_new_tokens, context_size, temperature = 0.0, top_k = None, eos_id = None):
    for _ in range(max_new_tokens):
        context = context[:, -context_size:]
        with torch.no_grad():
            logits = model(context)
            logits = logits[:, -1, :]
            if top_k:
                top_logits, _ = torch.topk(logits, k = top_k)
                min_logit = top_logits[:, -1]
                logits = torch.where(
                    logits < min_logit,
                    torch.tensor(float('-inf')).to(logits.device),
                    logits
                )
            if temperature > 0.0:
                logits = logits / temperature
                probs = torch.softmax(logits, dim = -1)
                next_token = torch.multinomial(probs, num_samples = 1)
            else:
                next_token = torch.argmax(logits, dim = -1, keepdim = True)
        if next_token == eos_id:
            break
        context = torch.cat((context, next_token), dim = 1)
    return token_ids_to_text(context)

In [35]:
text = generate_text(
    model = model,
    context = text_to_token_ids("I want to"),
    max_new_tokens = 10,
    context_size = gpt_config["context_length"],
    temperature = 0.1,
    top_k = 3
)
text

'I want toaleigh would gunned lightly rainRG� emailed ManhattanThomas'